# IOAI — 2025 Individual Contest Restroom (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2025"
CLONE = "IOAI-2025"
SUBDIR = "Individual-Contest/Restroom"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, SUBDIR))
# 대회 requirements.txt 의 관련 라이브러리 버전으로 정렬하는 constraints(아래 pip 에 적용)
CONSTRAINTS = r'''# IOAI-2025 대회 requirements.txt 에서 추출한 큐레이션 constraints — Colab (대회 핀 유지).
# torch/vllm/xformers/nvidia-cu12 는 제외(환경별 별도 관리).
# 생성: python -m ioai_env constraints
accelerate==1.8.1
audioread==3.0.1
bitsandbytes==0.46.1
cut-cross-entropy==25.1.1
datasets==3.6.0
einops==0.8.1
ftfy==6.3.1
hf-xet==1.1.5
hf_transfer==0.1.9
huggingface-hub==0.34.2
ipykernel==6.29.5
ipywidgets==7.8.5
jupyter_client==8.6.3
jupyter_core==5.7.2
jupyter_server==2.15.0
jupyterlab==4.3.4
librosa==0.11.0
matplotlib==3.10.0
msgspec==0.19.0
notebook==7.3.2
numpy==2.2.6
openai==1.90.0
opencv-python==4.11.0.86
opencv-python-headless==4.12.0.88
pandas==2.2.3
peft==0.16.0
pillow==11.1.0
protobuf==5.28.3
pydantic==2.10.3
regex==2024.11.6
safetensors==0.5.3
scikit-learn==1.6.1
scipy==1.13.1
seaborn==0.13.2
sentence-transformers==4.1.0
sentencepiece==0.2.0
soundfile==0.13.1
soxr==0.5.0.post1
timm==1.0.16
tokenizers==0.21.2
tqdm==4.67.1
traitlets==5.14.3
transformers==4.54.0
trl==0.19.1
tyro==0.9.26
unsloth==2025.7.8
unsloth_zoo==2025.7.10
'''
open('/content/ioai-constraints.txt', 'w').write(CONSTRAINTS)
!pip install -q -c /content/ioai-constraints.txt git+https://github.com/openai/CLIP.git ftfy regex
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

<img src="./figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2025 (Beijing, China), Individual Contest](https://ioai-official.org/china-2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Restroom/Restroom.ipynb)

# Restroom Icon Matching

## 1. Problem Description

This task focuses on training a model to learn the relationship between **male and female restroom icons from the same restroom**. Based on labeled training data, you are required to train a matching model that, given a query image, finds its **cross-gender counterpart** (e.g., match a male icon in its cropped image to its female counterpart in its original image), under the constraint that both icons come from the **same restroom**.

For example, for a query image (i.e., a male icon in cropped image) shown below in Fig1, its corresponding counterpart is shown in Fig2 (i.e., a female icon in original image which is from the same restroom).

| <img src="./figs/Restroom Fig 1.png" height="200"> | <img src="./figs/Restroom Fig 2.png" height="200"> |
| ------------------------------------------------------------ | ------------------------------------------------------------ |



## 2. Dataset

**(1) Training Set:** 

Used for model training. Directory structure:

```bash
train/
├── crop/         # Cropped icons
│   ├── female/   1.png, 2.png, ...
│   └── male/
└── orig/         # Original icons
    ├── female/
    └── male/
```

Each subfolder contains icons named from `1.png` to `82.png`, where the number indicates the restroom ID. For each restroom, there are four images (note that four images from the same restroom will share one same unique id across all four subfolders):

- `crop/female/i.png` → Cropped female icon  
- `crop/male/i.png`  → Cropped male icon  
- `orig/female/i.png` → Original female icon  
- `orig/male/i.png`  → Original male icon  

**(2) Validation Set and Test Set:** 

Either the validation set (test_a) or the test set (test_b) contains two subfolders:

```bash
test_a/           (or test_b/)
├── query/        # Cropped icons to be matched
└── gallery/      # Candidate original icons
```

- `query/`: **cropped** icons that need to be matched.  
- `gallery/`: pool of **original** icons to match from.  
- Note that:
    - Unlike the training split, the filenames in `query/` and `gallery/` are independently numbered and shuffled, which means that matching cannot rely on the ids
    - For each cropped icon in `query/` there are exactly two originals in `gallery/` (one male, one female from the same restroom), which means $\text{len}(\text{gallery})=2 * \text{len}(\text{query})$
- All images are in `.png` format. 
- The validation set has $10$ images in its `query/` folder, and the test set has $30$ images in its `query/` folder.

## 3. Task

For each image in the `query/` folder, predict the image in `gallery/` that:
- Is of the **opposite gender**, and
- Comes from the **same restroom**

This matching should be accomplished **using your trained model**. 

## 4. Submission Requirements

You must submit a notebook named `submission.ipynb`, which should include:

- Model training process (using `train/` data)  
- Matching process for both test sets (`test_a/` and `test_b/`)  
- The notebook must output two `.npy` files:
```bash
submission_a.npy     # Matching results for validation set (test_a).
submission_b.npy     # Matching results for test set (test_b).
```

Each npz file is a one-dimensional array with a size equal to the number of queries, and each value in the array corresponds to the image ID in the gallery.

For example, one valid `submission_a.npy` may looks like this:
<img src="./figs/Restroom Fig 3.png" width="300">

You can also find an example output format in [baseline.ipynb](https://ioai.bohrium.com/notebooks/81153159178).

## 5. Score

- If the submission finishes within the time limit, the score is calculated as:  
  $$
  \text{Score} = \frac{\text{Number of correct matches}}{\text{Total queries}}
  $$
  
  which will be a float number between $0.0$ and $1.0$ (inclusive).
  
- Submissions that exceed the time limit will receive a score of **0**.

## 6. Baseline and Training Set

- Below you can find the baseline solution.
- The dataset is in `training_set` folder. 
- The highest score by the Scientific Committee for this task is $0.90$ in Leaderboard B,  this score is used for score unification.
- The baseline score by the Scientific Committee for this task is $0.77$ in Leaderboard B,  this score is used for score unification.


In [ ]:
import random
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
import os
from PIL import Image
import torch.nn.functional as F

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
def sort_paths_by_number(path_list):
    """
    Sort based on the numerical values of the filenames in the path,
    assuming all filenames can be converted to integers.
    """
    def get_file_number(path):
        file_name = os.path.splitext(os.path.basename(path))[0]
        return int(file_name)

    path_list.sort(key=get_file_number)


In [ ]:
# import clip-vit model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
%pip install --quiet git+https://github.com/openai/CLIP.git
import clip

model, preprocess = clip.load("ViT-B/16", device=DEVICE)
model.eval()  # zero shot evaluation

In [ ]:
def infer(img_paths):
    """
    Compute L2‑normalized feature embeddings for a list of image file paths using the CLIP visual encoder.
    """
    # print(len(img_paths))  # for debug
    embeddings = []
    for path in tqdm(img_paths):
        img = Image.open(path)
        x = preprocess(img)
        x = x.type(torch.float16).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            emb = model.visual.forward(x)

        embeddings.append(emb)

    embeddings = torch.cat(embeddings)
    embeddings = F.normalize(embeddings, p=2, dim=1)

    return embeddings

In [ ]:
def match_images(BASE_DATA_DIR, result_path):
    """
    For each query image in BASE_DATA_DIR/query, find its best matching image
    in BASE_DATA_DIR/gallery by computing cosine similarity of CLIP embeddings,
    then save 1‑based match indices to result_path as a .npy file.
    """
    QUERY_DIR = BASE_DATA_DIR / "query"
    NON_QUERY_DIR = BASE_DATA_DIR / "gallery"

    query_image_paths = list(QUERY_DIR.glob("*.png"))
    non_query_image_paths = list(NON_QUERY_DIR.glob("*.png"))

    query_image_paths_str = [str(p) for p in query_image_paths]
    non_query_image_paths_str = [str(p) for p in non_query_image_paths]

    sort_paths_by_number(query_image_paths_str)
    sort_paths_by_number(non_query_image_paths_str)

    # print(query_image_paths_str)  # for debug

    query_embeddings = infer(query_image_paths_str)
    non_query_embeddings = infer(non_query_image_paths_str)
    distances = torch.mm(query_embeddings, non_query_embeddings.t())
    distances = (distances + 1.) / 2.

    topk_dists, topk_idxs = torch.topk(distances, 11, dim=1)  # distances has shape (num_queries, num_non_queries)

    topk_dists, topk_idxs = topk_dists.cpu(), topk_idxs.cpu()

    matches_dists, matches_idxs = topk_dists[:, 1], topk_idxs[:, 1]
    matches_dists = matches_dists.cpu().numpy()
    matches_idxs = matches_idxs.cpu().numpy()

    for i in range(len(matches_idxs)):
        matches_idxs[i]+=1

    # print(matches_idxs)  # for debug
    np.save(result_path, matches_idxs)  # save as .npy file

In [ ]:
"""the generated submission should be two .npy files"""
DATA_PATH = Path("Solution/")
OUTPUT_PATH = DATA_PATH / "Scoring"

match_images(DATA_PATH / "validation_set", OUTPUT_PATH / "submission_a.npy")
match_images(DATA_PATH / "test_set", OUTPUT_PATH / "submission_b.npy")

In [ ]:
import zipfile

files_to_zip = ['./Solution/Scoring/submission_a.npy', './Solution/Scoring/submission_b.npy']
zip_filename = 'submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created succefully!')

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission_a.npy', 'submission_b.npy']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)